# Machine Learnig in Raw Data 3-1

In [ ]:
# -*- coding: utf-8 -*-
# ====================================================================================
# FINAL LOUO - train on 2 users and predict the held-out user
# - Compact plots (no seaborn), letters outside collages
# - Complete artifacts (history, arrays, metrics.json/txt, predictions.csv)
# - Re-render from files
# - 1x3 collages by stem (holdout A/B/C)
# - Per-stem and aggregate tables (mean +/- std) with macro AUC from arrays
# ====================================================================================

from pathlib import Path
import sys
import re
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_curve, auc
)

#---------- Reproducibility Configuration
sys.path.insert(0, str(Path("src").resolve()))
from reproducibility import DEFAULT_SEED, configure_reproducibility
from plotting import (
    build_louo_vector_collages_for_stem,
    draw_confusion_matrix,
)

SEED = DEFAULT_SEED
configure_reproducibility(SEED, include_tensorflow=True)
print(f"Global reproducibility seed configured: {SEED}")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers


# ---------------- Figure style (publication quality, no seaborn)
mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.titlelocation": "left",
    "axes.titleweight": "regular",
    "axes.grid": False,
    "figure.autolayout": False,     # manual control with tight_layout/subplots_adjust
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif", "Nimbus Roman"]

# ---------------- Paths
BASE_DIR    = Path(".")
DATA_DIR    = BASE_DIR / "Data/3-Data-ML-1"
RESULTS_DIR = BASE_DIR / "Outputs/Results-Data-ML-3-1-v4-LOUO"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Collages by stem (LOUO)
LOUO_COLLAGES_DIR_ROOT = RESULTS_DIR / "_collages_louo"
LOUO_COLLAGES_DIR_ROOT.mkdir(parents=True, exist_ok=True)

# ---------------- Datasets
WINDOWS = [0.01, 0.02, 0.05, 0.15, 0.25, 0.50, 1.00, 2.00]
STEMS   = [f"ml_win_{w:.2f}s" for w in WINDOWS]
CSV_PATHS = {stem: DATA_DIR / f"{stem}.csv" for stem in STEMS}

# ---------------- Utilidades
RANDOM_STATE = SEED
configure_reproducibility(RANDOM_STATE, include_tensorflow=True)


def kernel_seed(offset: int):
    return keras.initializers.GlorotUniform(seed=RANDOM_STATE + offset)

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def _is_feature_col(c: str) -> bool:
    return bool(re.match(r"^S[1-7]_\d{5}$", str(c)))

def _feat_cols(df: pd.DataFrame):
    return [c for c in df.columns if _is_feature_col(c)]

def basic_info(df: pd.DataFrame, name: str) -> str:
    n_rows, n_cols = df.shape
    feat_cols = _feat_cols(df)
    users = df["user"].astype(str).unique().tolist()
    gestures = df["gesture"].astype(str).unique().tolist()
    vc_users = df["user"].astype(str).value_counts().sort_index()
    vc_gest  = df["gesture"].astype(str).value_counts().sort_index()
    return "\n".join([
        f"[{name}] shape = {n_rows} x {n_cols}",
        f"[{name}] #features = {len(feat_cols)}",
        f"[{name}] users = {users}",
        f"[{name}] gestures = {gestures}",
        f"[{name}] filas por user:\n{vc_users.to_string()}",
        f"[{name}] filas por gesture:\n{vc_gest.to_string()}",
    ])

# ---------------- Unit plots (compact, no titles)
def plot_training_curves(history_df: pd.DataFrame, out_base: Path):
    # Accuracy
    plt.figure(figsize=(5,5))
    if 'accuracy' in history_df:
        plt.plot(history_df['accuracy'].values, label="Training")
    if 'val_accuracy' in history_df:
        plt.plot(history_df['val_accuracy'].values, label="Validation", linestyle="--")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(loc="best"); plt.grid(True, alpha=0.3)
    plt.tight_layout(pad=0.2)
    p_png = out_base.parent / (out_base.name + "_accuracy.png")
    p_pdf = out_base.parent / (out_base.name + "_accuracy.pdf")
    plt.savefig(p_png, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.savefig(p_pdf, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.close()

    # Loss
    plt.figure(figsize=(5,5))
    if 'loss' in history_df:
        plt.plot(history_df['loss'].values, label="Training")
    if 'val_loss' in history_df:
        plt.plot(history_df['val_loss'].values, label="Validation", linestyle="--")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(loc="best"); plt.grid(True, alpha=0.3)
    plt.tight_layout(pad=0.2)
    p_png = out_base.parent / (out_base.name + "_loss.png")
    p_pdf = out_base.parent / (out_base.name + "_loss.pdf")
    plt.savefig(p_png, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.savefig(p_pdf, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.close()

def _nice_confusion(ax, cm: np.ndarray, class_names):
    draw_confusion_matrix(ax, cm, list(class_names))

def plot_confusion_from_arrays(y_true_int, y_pred_int, class_names, out_path: Path):
    labels = list(range(len(class_names)))
    cm = confusion_matrix(y_true_int, y_pred_int, labels=labels)
    fig, ax = plt.subplots(figsize=(6,5))
    _nice_confusion(ax, cm, class_names)
    plt.tight_layout(pad=0.2)
    png = out_path.parent / (out_path.name + ".png")
    pdf = out_path.parent / (out_path.name + ".pdf")
    plt.savefig(png, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.savefig(pdf, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)

def plot_roc_from_arrays(y_true_int, y_prob, class_names, out_path: Path):
    n_classes = len(class_names)
    y_true_ohe = np.eye(n_classes)[y_true_int]
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_ohe[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_ohe.ravel(), y_prob.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes): mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    plt.figure(figsize=(7,6))
    plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"micro (AUC={roc_auc['micro']:.3f})")
    plt.plot(fpr["macro"], tpr["macro"], linestyle="--", label=f"macro (AUC={roc_auc['macro']:.3f})")
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})")
    plt.plot([0,1], [0,1], "k:", alpha=0.5)
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right", fontsize=8); plt.grid(True, alpha=0.3)
    plt.tight_layout(pad=0.2)
    png = out_path.parent / (out_path.name + ".png")
    pdf = out_path.parent / (out_path.name + ".pdf")
    plt.savefig(png, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.savefig(pdf, dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.close()

# ---------------- Collages (compact, letters outside)
def _letters_seq(n):
    letters = []
    i = 0
    while len(letters) < n:
        s = ""
        k = i
        while True:
            s = chr(ord('A') + (k % 26)) + s
            k = k // 26 - 1
            if k < 0:
                break
        letters.append(s)
        i += 1
    return letters

def _compose_image_grid(*args, **kwargs):
    raise RuntimeError("Use vector collage builders based on saved arrays and histories.")

# ---------------- Modelo
def build_model(input_dim: int, n_classes: int) -> keras.Model:
    l2 = regularizers.l2(1e-4)
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(1024, activation="relu", kernel_regularizer=l2, kernel_initializer=kernel_seed(1)),
        layers.Dropout(0.3, seed=RANDOM_STATE + 101),
        layers.Dense(512, activation="relu", kernel_regularizer=l2, kernel_initializer=kernel_seed(2)),
        layers.Dropout(0.3, seed=RANDOM_STATE + 102),
        layers.Dense(128, activation="relu", kernel_regularizer=l2, kernel_initializer=kernel_seed(3)),
        layers.Dropout(0.2, seed=RANDOM_STATE + 103),
        layers.Dense(n_classes, activation="softmax", kernel_initializer=kernel_seed(4)),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---------------- Macro AUC from saved arrays
def _macro_auc_from_files(exp_dir: Path) -> float:
    try:
        with open(exp_dir / "classes.json", "r", encoding="utf-8") as f:
            classes = json.load(f)["classes"]
        y_true = pd.read_csv(exp_dir / "arrays" / "y_true.csv")["y_true"].values
        y_prob = pd.read_csv(exp_dir / "arrays" / "y_prob.csv").values
        n_classes = len(classes)
        y_true_ohe = np.eye(n_classes)[y_true]
        aucs = []
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_ohe[:, i], y_prob[:, i])
            aucs.append(auc(fpr, tpr))
        return float(np.mean(aucs)) if len(aucs) else np.nan
    except Exception:
        return float("nan")

# ---------------- LOUO core
def run_louo_for_stem(stem: str, df: pd.DataFrame):
    feat_cols = _feat_cols(df)
    if not feat_cols:
        print(f"[WARN] {stem}: no S#_xxxxx feature columns found. Skipped.")
        return

    stem_dir = RESULTS_DIR / stem
    ensure_dir(stem_dir)
    with open(stem_dir / "dataset_info.txt", "w", encoding="utf-8") as f:
        f.write(basic_info(df, stem) + "\n")

    users_all = sorted(df["user"].astype(str).unique().tolist())
    cross_logs = []

    for holdout_user in users_all:
        out_dir    = stem_dir / "cross_user" / f"holdout_user_{holdout_user}"
        plots_dir  = out_dir / "plots"
        arrays_dir = out_dir / "arrays"
        ensure_dir(plots_dir); ensure_dir(arrays_dir)

        # User split (train = all except holdout; test = holdout)
        df_train = df[df["user"].astype(str) != holdout_user].reset_index(drop=True)
        df_test  = df[df["user"].astype(str) == holdout_user].reset_index(drop=True)
        if df_train.empty or df_test.empty:
            print(f"[WARN] {stem}|holdout={holdout_user}: empty split. Skipped."); 
            continue

        # X/y (encoder fitted on train only)
        feat_cols = _feat_cols(df_train)
        X_train = df_train[feat_cols].to_numpy(dtype=np.float32)
        X_test  = df_test[feat_cols].to_numpy(dtype=np.float32)

        le = LabelEncoder()
        y_train_g = df_train["gesture"].astype(str).values
        y_train   = le.fit_transform(y_train_g)
        y_test_g  = df_test["gesture"].astype(str).values

        # Filtrar clases no vistas en train (evita ValueError al transformar)
        mask_known = np.isin(y_test_g, le.classes_)
        if not mask_known.all():
            X_test  = X_test[mask_known]
            y_test_g = y_test_g[mask_known]
            df_test  = df_test.loc[mask_known].reset_index(drop=True)
        y_test = le.transform(y_test_g)
        n_classes = len(le.classes_)

        # Split Train/Val (estratificar por user||gesture → user → gesture si se puede)
        def _min_needed_for_test(test_size: float) -> int:
            return max(2, int(np.ceil(1.0/float(test_size))))
        def safe_stratify_vector(vec: pd.Series, test_size=0.1):
            vc = vec.value_counts(); mn = _min_needed_for_test(test_size)
            if (vc < mn).any() or vc.shape[0] < 2: return None
            return vec

        candidates = [
            ("user_gesture", df_train["user"].astype(str) + "||" + df_train["gesture"].astype(str)),
            ("user", df_train["user"].astype(str)),
            ("gesture", pd.Series(y_train))
        ]
        strat, strat_used = None, None
        for name, vec in candidates:
            s = safe_stratify_vector(pd.Series(vec), test_size=0.1)
            if s is not None: strat, strat_used = s, name; break

        tr_idx, val_idx = train_test_split(
            np.arange(len(df_train)), test_size=0.1, random_state=RANDOM_STATE,
            stratify=strat if strat_used else None
        )
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        # Standardization
        scaler = StandardScaler(with_mean=True, with_std=True)
        X_tr_s  = scaler.fit_transform(X_tr)
        X_val_s = scaler.transform(X_val)
        X_te_s  = scaler.transform(X_test)

        # Modelo
        configure_reproducibility(RANDOM_STATE, include_tensorflow=True)
        model = build_model(X_tr_s.shape[1], n_classes)
        callbacks = [
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1)
        ]

        # Train
        t0_fit = time.perf_counter()
        history = model.fit(
            X_tr_s, y_tr,
            epochs=100, batch_size=16,
            validation_data=(X_val_s, y_val),
            shuffle=True,
            verbose=0, callbacks=callbacks
        )
        t_fit = time.perf_counter() - t0_fit
        epochs_trained = len(history.history.get("loss", []))

        # Save history.csv and classes
        hist_df = pd.DataFrame({
            "epoch": np.arange(1, epochs_trained+1),
            "accuracy": history.history.get("accuracy", [None]*epochs_trained),
            "val_accuracy": history.history.get("val_accuracy", [None]*epochs_trained),
            "loss": history.history.get("loss", [None]*epochs_trained),
            "val_loss": history.history.get("val_loss", [None]*epochs_trained),
        })
        hist_df.to_csv(out_dir / "history.csv", index=False)
        with open(out_dir / "classes.json", "w", encoding="utf-8") as f:
            json.dump({"classes": list(le.classes_)}, f, ensure_ascii=False, indent=2)

        # Predecir TEST
        t0_test = time.perf_counter()
        prob_te = model.predict(X_te_s, verbose=0)
        t_test  = time.perf_counter() - t0_test
        y_pred  = np.argmax(prob_te, axis=1)

        # Save arrays
        pd.DataFrame({"y_true": y_test}).to_csv(arrays_dir / "y_true.csv", index=False)
        pd.DataFrame({"y_pred": y_pred}).to_csv(arrays_dir / "y_pred.csv", index=False)
        pd.DataFrame(prob_te, columns=[f"p_{c}" for c in le.classes_]).to_csv(arrays_dir / "y_prob.csv", index=False)

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
            y_test, y_pred, average="macro", zero_division=0)
        prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
            y_test, y_pred, average="weighted", zero_division=0)
        cls_report = classification_report(y_test, y_pred, target_names=list(le.classes_), zero_division=0)

        # Graficar
        plot_training_curves(hist_df, plots_dir / "training_curves")
        plot_confusion_from_arrays(y_test, y_pred, list(le.classes_), plots_dir / "confusion_matrix")
        plot_roc_from_arrays(y_test, prob_te, list(le.classes_), plots_dir / "roc_curves")

        # JSON + TXT
        metrics_payload = {
            "experiment": f"{stem} — LOUO holdout={holdout_user}",
            "stem": stem, "holdout_user": holdout_user,
            "train_users": sorted(list(set(users_all) - {holdout_user})),
            "train_shape": [int(X_tr_s.shape[0]), int(X_tr_s.shape[1])],
            "val_shape":   [int(X_val_s.shape[0]), int(X_val_s.shape[1])],
            "test_shape":  [int(X_te_s.shape[0]), int(X_te_s.shape[1])],
            "stratify_train_val": strat_used,
            "timings": {"train_time_s": float(t_fit), "test_time_s": float(t_test), "epochs_trained": int(epochs_trained)},
            "metrics": {
                "accuracy": round(float(acc),4),
                "precision_macro": round(float(prec_macro),4),
                "recall_macro": round(float(rec_macro),4),
                "f1_macro": round(float(f1_macro),4),
                "precision_weighted": round(float(prec_w),4),
                "recall_weighted": round(float(rec_w),4),
                "f1_weighted": round(float(f1_w),4)
            },
            "classes": list(le.classes_)
        }
        with open(out_dir / "metrics.json", "w", encoding="utf-8") as f:
            json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

        with open(out_dir / "metrics.txt", "w", encoding="utf-8") as f:
            f.write(f"Experiment: {stem} — HOLDOUT user={holdout_user}\n")
            f.write(f"Train users: {sorted(set(users_all) - {holdout_user})}\n")
            f.write(f"Train shape: {X_tr_s.shape} | Val shape: {X_val_s.shape} | Test shape: {X_te_s.shape}\n")
            f.write(f"Stratify train/val usado: {strat_used}\n")
            f.write("\n== Timings (s) ==\n")
            f.write(f"train_time_s: {t_fit:.6f}\n")
            f.write(f"test_time_s:  {t_test:.6f}\n")
            f.write(f"epochs_trained: {epochs_trained}\n")
            f.write("\n== Metrics (TEST) ==\n")
            f.write(f"Accuracy: {acc:.4f}\n")
            f.write(f"Precision (macro): {prec_macro:.4f}\n")
            f.write(f"Recall (macro): {rec_macro:.4f}\n")
            f.write(f"F1-score (macro): {f1_macro:.4f}\n")
            f.write(f"Precision (weighted): {prec_w:.4f}\n")
            f.write(f"Recall (weighted): {rec_w:.4f}\n")
            f.write(f"F1-score (weighted): {f1_w:.4f}\n")
            f.write("\n== Classification report (TEST) ==\n")
            f.write(cls_report)

        # Prediction CSV
        pred_df = pd.DataFrame({
            "user": df_test["user"].astype(str).values,
            "gesture_true": y_test_g,
            "gesture_pred": le.inverse_transform(y_pred),
            "prob_max": prob_te.max(axis=1)
        })
        pred_df.to_csv(out_dir / "predictions.csv", index=False)

        # Log summary by holdout
        auc_macro = _macro_auc_from_files(out_dir)
        cross_logs.append({
            "stem": stem, "holdout_user": holdout_user,
            "n_train": int(X_tr_s.shape[0]), "n_val": int(X_val_s.shape[0]), "n_test": int(X_te_s.shape[0]),
            "accuracy": acc, "precision_macro": prec_macro, "recall_macro": rec_macro, "f1_macro": f1_macro,
            "precision_weighted": prec_w, "recall_weighted": rec_w, "f1_weighted": f1_w,
            "auc_macro": auc_macro,
            "train_time_s": t_fit, "test_time_s": t_test, "epochs_trained": epochs_trained,
            "stratify_train_val": strat_used
        })

    # Summary by stem + 1x3 collages
    if cross_logs:
        sum_df = pd.DataFrame(cross_logs)
        sum_df.to_csv(stem_dir / "louo_summary.csv", index=False)
        _build_louo_collages_for_stem(stem)

# ---------------- LOUO 1x3 collages by stem
def _build_louo_collages_for_stem(stem: str):
    return build_louo_vector_collages_for_stem(RESULTS_DIR, stem)

# ---------------- Re-render LOUO from files
def _rerender_louo_plots_from_saved_files():
    for stem in STEMS:
        stem_dir = RESULTS_DIR / stem / "cross_user"
        if not stem_dir.exists():
            continue
        for udir in sorted(stem_dir.glob("holdout_user_*")):
            if not (udir / "history.csv").exists():
                continue
            # Cargar clases e history
            with open(udir / "classes.json", "r", encoding="utf-8") as f:
                classes = json.load(f)["classes"]
            hist_df = pd.read_csv(udir / "history.csv")
            plot_training_curves(hist_df, udir / "plots" / "training_curves")
            # Arrays
            y_true = pd.read_csv(udir / "arrays" / "y_true.csv")["y_true"].values
            y_pred = pd.read_csv(udir / "arrays" / "y_pred.csv")["y_pred"].values
            y_prob = pd.read_csv(udir / "arrays" / "y_prob.csv").values
            plot_confusion_from_arrays(y_true, y_pred, classes, udir / "plots" / "confusion_matrix")
            plot_roc_from_arrays(y_true, y_prob, classes, udir / "plots" / "roc_curves")
        _build_louo_collages_for_stem(stem)

# ---------------- LOUO tables (by holdout and aggregated by dt)
def _collect_louo_rows():
    rows = []
    for stem in STEMS:
        stem_dir = RESULTS_DIR / stem / "cross_user"
        if not stem_dir.exists():
            continue
        for udir in sorted(stem_dir.glob("holdout_user_*")):
            mpath = udir / "metrics.json"
            if not mpath.exists():
                continue
            with open(mpath, "r", encoding="utf-8") as f:
                m = json.load(f)
            auc_macro = _macro_auc_from_files(udir)
            dt = stem.replace("ml_win_", "").replace("s", "")
            rows.append({
                "Δt (s)": float(dt),
                "Holdout": udir.name.replace("holdout_user_", ""),
                "Accuracy": m["metrics"]["accuracy"],
                "F1 (macro)": m["metrics"]["f1_macro"],
                "AUC (macro)": auc_macro,
                "stem": stem
            })
    return rows

def _print_and_save_louo_tables():
    rows = _collect_louo_rows()
    tables_dir = RESULTS_DIR / "tables"
    ensure_dir(tables_dir)

    if not rows:
        print("\n[WARN] No LOUO metrics found to build tables.")
        return

    df = pd.DataFrame(rows)
    df = df.sort_values(["Δt (s)", "Holdout"]).reset_index(drop=True)
    for col in ["Accuracy", "F1 (macro)", "AUC (macro)"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").round(4)

    print("\nTabla LOUO — por window y holdout")
    print(df.drop(columns=["stem"]).to_string(index=False, formatters={"Δt (s)": lambda x: f"{x:.2f}"}))
    df.to_csv(tables_dir / "louo_by_holdout.csv", index=False)
    with open(tables_dir / "louo_by_holdout.tex", "w", encoding="utf-8") as f:
        f.write(df.drop(columns=["stem"]).to_latex(index=False, float_format="%.4f"))

    # Agregado por window (mean ± std)
    agg = df.groupby("Δt (s)")[
        ["Accuracy", "F1 (macro)", "AUC (macro)"]
    ].agg(["mean", "std"]).reset_index()
    # Aplanar columnas
    agg.columns = ["Δt (s)"] + [f"{m} {stat}" for m in ["Accuracy", "F1 (macro)", "AUC (macro)"] for stat in ["mean", "std"]]
    agg = agg.sort_values("Δt (s)").reset_index(drop=True)
    agg_rounded = agg.copy()
    for c in agg.columns[1:]:
        agg_rounded[c] = pd.to_numeric(agg[c], errors="coerce").round(4)

    print("\nTabla LOUO — agregada por window (mean ± std)")
    print(agg_rounded.to_string(index=False, formatters={"Δt (s)": lambda x: f"{x:.2f}"}))
    agg_rounded.to_csv(tables_dir / "louo_agg_by_window.csv", index=False)

    # LaTeX version with mean +/- std in one cell
    def fmt_mean_std(m, s): 
        try:
            return f"{m:.4f} ± {s:.4f}"
        except Exception:
            return ""
    tex_df = pd.DataFrame({
        "Δt (s)": agg["Δt (s)"].map(lambda x: f"{x:.2f}"),
        "Accuracy": [fmt_mean_std(m, s) for m, s in zip(agg["Accuracy mean"], agg["Accuracy std"])],
        "F1 (macro)": [fmt_mean_std(m, s) for m, s in zip(agg["F1 (macro) mean"], agg["F1 (macro) std"])],
        "AUC (macro)": [fmt_mean_std(m, s) for m, s in zip(agg["AUC (macro) mean"], agg["AUC (macro) std"])],
    })
    with open(tables_dir / "louo_agg_by_window.tex", "w", encoding="utf-8") as f:
        f.write(tex_df.to_latex(index=False, escape=True))

# ---------------- MAIN
def main():
    print("Reading datasets from:", DATA_DIR.resolve())
    loaded_any = False
    available_stems = []

    for stem, csv_path in CSV_PATHS.items():
        if not csv_path.exists():
            print(f"[WARN] Not found: {csv_path}")
            continue

        df = pd.read_csv(csv_path, dtype={"user": str, "gesture": str})
        if "user" not in df.columns or "gesture" not in df.columns:
            print(f"[WARN] {stem}: missing columns 'user' o 'gesture'. Skipped.")
            continue

        df = df.sort_values(["user", "gesture"]).reset_index(drop=True)
        run_louo_for_stem(stem, df)
        available_stems.append(stem)
        loaded_any = True

    if not loaded_any:
        print("\nNo dataset was loaded. Check the folder:", DATA_DIR.resolve())
        return

    # Re-render from files and recreate collages
    _rerender_louo_plots_from_saved_files()
    # LOUO tables (by holdout and aggregated by window)
    _print_and_save_louo_tables()
    print("\nDone. Check results in:", RESULTS_DIR.resolve())

if __name__ == "__main__":
    main()
# ====================================================================================
